# Secure ViT – Colab notebook

All runs are triggered from this notebook. Service code lives in `*.py` modules; this notebook drives setup, data loading, training, and inference.

## 1. Setup (run first)

Upload this project to Colab (e.g. zip and upload, or clone from Git), then set `PROJECT_ROOT` to the path that contains the `data` folder. Install dependencies and add project to path.

In [ ]:
from pathlib import Path
import sys

# Path to project root (contains data/, requirements.txt, etc.)
# If you uploaded a zip: unzip and set this to the extracted folder.
PROJECT_ROOT = Path("/content/Secure-Inference-Token-Reduced-VIT")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path(".").resolve()  # fallback: current dir

sys.path.insert(0, str(PROJECT_ROOT))
%pip install -q -r "{0}".format(PROJECT_ROOT / 'requirements.txt')
print("Project root:", PROJECT_ROOT)

## 2. Kaggle authentication (for dataset)

To download the LC25000 dataset via kagglehub, add your Kaggle API key. In Colab: Secrets (key icon) → add `KAGGLE_USERNAME` and `KAGGLE_KEY`, or upload `~/.kaggle/kaggle.json`.

In [ ]:
# Optional: if you uploaded kaggle.json to Colab, uncomment and run:
# !mkdir -p ~/.kaggle
# !cp /content/kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

## 3. Load dataset (train / val / test)

Download LC25000 via kagglehub and build DataLoaders. Test set is held out for final inference (different images from train/val).

In [ ]:
from data import get_lc25000_root, get_dataloaders

DATA_ROOT = get_lc25000_root()  # downloads if needed
print("Dataset root:", DATA_ROOT)

BATCH_SIZE = 32
VAL_RATIO = 0.15
TEST_RATIO = 0.15   # ~70% train, 15% val, 15% test
SEED = 42
IMAGE_SIZE = 224

train_loader, val_loader, test_loader = get_dataloaders(
    root_dir=DATA_ROOT,
    batch_size=BATCH_SIZE,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED,
    subdir_depth=2,   # LC25000: lung_image_sets/..., colon_image_sets/...
    image_size=IMAGE_SIZE,
    num_workers=2,
)

train_ds = train_loader.dataset
print("Classes:", train_ds.class_names)
print("Train:", len(train_ds), "| Val:", len(val_loader.dataset), "| Test:", len(test_loader.dataset))

## 4. Dataset summary

Verify split sizes and that test set is disjoint (for inference evaluation).

In [ ]:
import collections

for name, loader in [("Train", train_loader), ("Val", val_loader), ("Test", test_loader)]:
    ds = loader.dataset
    counts = collections.Counter(lbl for _, lbl in ds.samples)
    print(f"{name}: n={len(ds)}, batches={len(loader)}")
    print(f"  Per class: {dict(sorted(counts.items(), key=lambda x: x[0]))}")

## 5. (Optional) Inspect one batch

Sanity check: show a few images and labels from the train loader.

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(iter(train_loader))
class_names = train_loader.dataset.class_names

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    if i < images.shape[0]:
        img = images[i].permute(1, 2, 0).numpy()
        img = (img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]).clip(0, 1)
        ax.imshow(img)
        ax.set_title(class_names[labels[i].item()], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Next steps

Later cells will trigger: teacher training, student distillation, and encrypted inference. Use `train_loader` / `val_loader` for training and `test_loader` for final evaluation and inference.